In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pymysql

load_dotenv(override=True)
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
) 
MODEL_NAME = "gpt-5-nano"

In [ ]:
import pymysql
import json
def get_weather(city):
    return {
        "城市": city,
        "天气": "晴",
        "温度": "25°C",
        "湿度": "60%",
        "风速": "5 km/h"
    } 

def send_msg(message):
    """
    发送天气提醒给用户
    """
    
    return {
        "status": "success",
        "message": f"已发送消息: {message}"
    }


def sql_query(sql_statement):
    """
    查询本地MySQL数据库，执行一段SQL代码，并返回查询结果
    """
    connection = pymysql.connect(
        host='localhost',
        user='root',
        password='000000',
        database='pk',
        charset='utf8'
    )

    try:
        with connection.cursor() as cursor:
            cursor.execute(sql_statement)
            result = cursor.fetchall()
    finally:
        connection.close()
    
    return json.dumps(result, ensure_ascii=True)  # 将查询结果转换成JSON字符串格式返回


   

# 将工具函数封装成符合规范的工具描述
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气信息，包括：天气、温度、湿度和风速等；当用户询问天气时，应该调用这个工具；",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "要查询的城市名称，例如：北京",
                    }
                },
                "required": ["city"]
            },
        }
    },
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "send_msg",
    #         "description": "发送天气提醒给用户",
    #         "parameters": {
    #             "type": "object",
    #             "properties": {
    #                 "message": {
    #                     "type": "string",
    #                     "description": "要发送的消息内容",
    #                 }
    #             },
    #             "required": ["message"]
    #         },
    #     }
    # },
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "sql_query",
    #         "description": "查询本地MySQL数据库，执行一段SQL代码，并返回查询结果",
    #         "parameters": {
    #             "type": "object",
    #             "properties": {
    #                 "sql_statement": {
    #                     "type": "string",
    #                     "description": "字符串形式的SQL查询语句，用于执行对MySQL数据库中pk库进行查询",
    #                 }
    #             },
    #             "required": ["sql_statement"]
    #         },
    #     }
    # },
]

available_functions = {
    "get_weather": get_weather,
    "send_msg": send_msg,
    "sql_query": sql_query
}

In [ ]:
def test_tool_choice(prompt:str, mode):
    print(f"=== Testing {mode} tool choice ===")
    print(f"Prompt: {prompt}")

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        tools=tools,  # 天气、消息发送、mysql查询三种工具
        tool_choice=mode,
        temperature=0.7
    )
    msg = response.choices[0].message


    print(response.choices[0].finish_reason)

    if msg.tool_calls:
        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            arguments = tool_call.function.arguments
            print(f"调用工具: {tool_name}， 参数: {arguments}")
    else:
        print(f"直接回答：{msg.content}")

In [ ]:
prompt = "什么是大模型"
test_tool_choice(prompt, mode="required")